In [ ]:
!date

In [ ]:
### imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

In [ ]:
### paths
projdir = '/u/project/cluo/terencew/claude/project_ideas/asm_lr_hprc2'
outdir = f'{projdir}/results/qc'

# QC comparisons across superpopulation/PCLAI/individual, for supplemental figures -- checking
# these are comparable before any downstream ASM analysis leans on them. Not ASM-specific.
# n_reads/read_length/qscore come from the basecaller's own sequencing_summary.txt.gz (pre-
# haplotype-split, authoritative); hap1/hap2_coverage come from this project's own modbed parse
# (aligned span / assembly contig size) since the basecaller file has no haplotype assignment.
# See scripts/harmonize_hg38/pilot/P04_read_length_coverage_qc.py for the two-source rationale
# -- superseded by notebooks/qc/QC02_hprc2_supp_seq_qc.ipynb now that the paper's own
# Supplementary Table 6 is in hand (P04 kept as a low-priority cross-validation option, not
# removed).

In [ ]:
### load manifest
manifest = pd.read_csv(f'{projdir}/tsv/meta/hprc2_sample_manifest.tsv', sep='\t')

In [ ]:
manifest.head()

In [ ]:
manifest.shape

In [ ]:
### pilot donors -- P04 has only been run on these 2 so far (both in asm_lr's 18-donor cohort
### too, for cross-project comparability). Expand samples list once P04 is run as an SGE array
### across all 229 (not yet submitted -- see scripts/qsub/ convention used elsewhere in this
### project, none written for P04 yet).
pilot_samples = ['HG00126', 'HG00146']

In [ ]:
### load read-length / coverage QC (scripts/pilot/P04_read_length_coverage_qc.py output)
qc = pd.read_csv(f'{outdir}/data/read_length_coverage_qc.tsv', sep='\t')

In [ ]:
qc.head()

In [ ]:
qc.shape

In [ ]:
### PCLAI per-sample summary -- mean PC1/PC2 across genome-wide windows, per haplotype.
### crude on purpose (a mean over a continuous ancestry axis, not a discretized label) --
### enough to check the cohort spans the intended ancestry range, not a real ancestry analysis.
def load_pclai(args):
    sample, hap = args
    path = f'{projdir}/tsv/meta/pclai/{sample}_hap{hap}_pclai_v1.1.grch38_coord.bed'
    df = pd.read_csv(path, sep='\t', header=None)
    coords = df[9].str.strip('()').str.split(',', expand=True).astype(float)
    return sample, hap, coords[0].mean(), coords[1].mean()

pclai_tasks = [(s, h) for s in pilot_samples for h in (1, 2)]
pclai_rows = []
with ProcessPoolExecutor(max_workers=4) as ex:
    for sample, hap, pc1, pc2 in tqdm(ex.map(load_pclai, pclai_tasks), total=len(pclai_tasks)):
        pclai_rows.append((sample, hap, pc1, pc2))
pclai = pd.DataFrame(pclai_rows, columns=['sample', 'hap', 'pclai_pc1_mean', 'pclai_pc2_mean'])

In [ ]:
pclai.head()

In [ ]:
pclai.shape

In [ ]:
### merge QC + manifest (population/superpopulation) + PCLAI (wide, one column per hap)
pclai_wide = pclai.pivot(index='sample', columns='hap', values=['pclai_pc1_mean', 'pclai_pc2_mean'])
pclai_wide.columns = [f'{a}_hap{b}' for a, b in pclai_wide.columns]
pclai_wide = pclai_wide.reset_index()

merged = qc.merge(manifest[['sample_id', 'population', 'superpopulation']],
                   left_on='sample', right_on='sample_id', how='left')
merged = merged.merge(pclai_wide, on='sample', how='left')
merged = merged.drop(columns=['sample_id'])

In [ ]:
merged.head()

In [ ]:
merged.shape

In [ ]:
### QC comparison figures -- only 2 donors right now (n too small to mean anything by
### superpopulation yet), this just proves the plotting/merge shape works before scaling P04
### to the full 229-sample manifest
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].bar(merged['sample'], merged['median_read_length'])
axes[0].set_ylabel('median read length (bp)')

axes[1].bar(merged['sample'], (merged['hap1_coverage'] + merged['hap2_coverage']) / 2)
axes[1].set_ylabel('mean haplotype coverage (x)')

axes[2].scatter(merged['pclai_pc1_mean_hap1'], merged['pclai_pc2_mean_hap1'])
for _, row in merged.iterrows():
    axes[2].annotate(row['sample'], (row['pclai_pc1_mean_hap1'], row['pclai_pc2_mean_hap1']))
axes[2].set_xlabel('PCLAI PC1 (hap1, genome mean)')
axes[2].set_ylabel('PCLAI PC2 (hap1, genome mean)')

plt.tight_layout()

In [ ]:
### write output
merged.to_csv(f'{outdir}/data/qc_pclai_merged.tsv', sep='\t', index=False)
fig.savefig(f'{outdir}/figures/qc_overview_pilot.png', dpi=150)

In [ ]:
!date